# Process Simulation Notebook
This notebook is just to hold the values that we get from the `omp_baseline`.

# Checking the size of all files:
``` bash
brand@brandon-x13 MINGW64 /c/Users/brand/Documents/engs109-omp-fpga/SW/processor_simulation
$ wc -c *.bin
  1024 col_norms.bin
131072 D.bin
262144 Psi.bin
122880 X.bin
 61440 Y.bin
578560 total
```

# Verification the the `omp_baseline.c` file matches with the simulation.
``` bash
brand@brandon-x13 MINGW64 /c/Users/brand/Documents/engs109-omp-fpga/SW/processor_simulation
$ ./omp_baseline.exe 120
windows           : 120
correlation passes: 3840
mean PRD          : 20.1318 %
mean per-pass time: 19.0713 us  <-- divide HDL-A cycles*10ns into this
per-window results written to baseline_results.csv
```

# Phase 1: Get the data on-chip without a filesystem
Bare metal Zynq has no `fopen`, so `load_f32` won't work there. We don't need all 120 windows for timing. We need the dictionary and a couple of representative residuals. Generate a C header from the notebook.

In [4]:
import numpy as np
M_LEN, N_ATOMS, WINDOW_N = 128, 256, 256
# D.bin is already atom-major (you exported D_float.T), so reshape to (N_ATOMS, M_LEN)
D_atom_major = np.fromfile("D.bin", dtype="<f4").reshape(N_ATOMS, M_LEN)
Y = np.fromfile("Y.bin", dtype="<f4").reshape(-1, M_LEN)   # all windows

def dump_c_array(name, arr):
    flat = np.asarray(arr, dtype="<f4").ravel()
    return f"const float {name}[{flat.size}] = {{\n" + \
           ",".join(repr(float(v)) + "f" for v in flat) + "\n};"

with open("baseline_data.h", "w") as f:
    f.write("#ifndef BASELINE_DATA_H\n#define BASELINE_DATA_H\n\n")
    f.write(dump_c_array("D_data", D_atom_major) + "\n\n")   # already atom-major — NO .T here
    f.write(dump_c_array("r_data", Y[0]) + "\n\n")
    f.write("#endif\n")
print("wrote baseline_data.h  | D_data:", D_atom_major.size, " r_data:", Y.shape[1])

wrote baseline_data.h  | D_data: 32768  r_data: 128


# Phase 2: Build the `.xsa` in Vivado and make a Platform Project in Vitis

# Phase 3: Vitis Application Project

## Results
``` bash
BUILD_V3 sink=23230  last_val=19.759178
total=187120.455574 us over 1000 reps -> per-pass: 187.120456 us
BUILD_V4 sink=23230  last_val=19.759178
total=187130.751573 us over 1000 reps -> per-pass: 187.130752 us
BUILD_V5 sink=23230  last_val=19.759178
total=187120.299574 us over 1000 reps -> per-pass: 187.120300 us
```
### Interpretation:
~187.1 µs per correlation pass (mean of 3 runs: 187.124 µs) Cortex-A9 @ 667 MHz, tuned C (-O3, -mfpu=neon, -mfloat-abi=hard, -ffast-math) Single COMPUTE pass: 256 atoms × 128-tap inner product + argmax Compute-only, AXI transfer excluded
